In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.datasets import mnist
import numpy as np

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
#Normaliser les images entre 0 et 1 et les aplatir (28x28 -> 784)

In [4]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape((x_train.shape[0], -1))  # (60000, 784)
x_test = x_test.reshape((x_test.shape[0], -1))  # (10000, 784)

In [5]:
 #Construire l'auto-encodeur
input_img = layers.Input(shape=(784,))
encoded = layers.Dense(128, activation='relu')(input_img)  # Encoder: reduction de dimension
decoded = layers.Dense(784, activation='sigmoid')(encoded)  # Decoder: reconstruction


In [6]:
autoencoder = models.Model(input_img, decoded)
encoder = models.Model(input_img, encoded)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [7]:
#Entrainer l'auto-encodeur
autoencoder.fit(x_train, x_train, epochs=10, batch_size=256, validation_data=(x_test, x_test))


Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.3140 - val_loss: 0.1331
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.1245 - val_loss: 0.1019
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.0991 - val_loss: 0.0883
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 0.0873 - val_loss: 0.0811
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0806 - val_loss: 0.0769
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - loss: 0.0769 - val_loss: 0.0741
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 0.0742 - val_loss: 0.0724
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.0725 - val_loss: 0.0710
Epoch 9/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0712 - val_loss: 0.0700
Epoch 10/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - loss: 0.0704 - val_loss: 0.0693


In [8]:
#Extraire les representations latentes
x_train_latent = encoder.predict(x_train)
x_test_latent = encoder.predict(x_test)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [9]:
#Entrainer un classificateur Logistic Regression sur les representations latentes
clf_latent = LogisticRegression(max_iter=1000)
clf_latent.fit(x_train_latent, y_train)

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [10]:
#Predictions et evaluation sur les vecteurs latents
y_pred_latent = clf_latent.predict(x_test_latent)
accuracy_latent = accuracy_score(y_test, y_pred_latent)
print(f"Precision avec les vecteurs latents : {accuracy_latent * 100:.2f}%")

Precision avec les vecteurs latents : 92.53%


In [11]:
#Entrainer un classificateur Logistic Regression sur les images originales
clf_original = LogisticRegression(max_iter=1000)
clf_original.fit(x_train, y_train)

LogisticRegression(max_iter=1000)

In [12]:
#Predictions et evaluation sur les images originales
y_pred_original = clf_original.predict(x_test)
accuracy_original = accuracy_score(y_test, y_pred_original)
print(f"Precision avec les images originales : {accuracy_original * 100:.2f}%")

Precision avec les images originales : 92.64%
